<a href="https://colab.research.google.com/github/sruthi-analyst/sruthi-codeboosters-2026/blob/main/Day2/Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt #data visualisation library
import warnings                 #errors might come for versions and other things, so we import warnings and then tell the system to ignore them all for smooth functioning of the program
warnings.filterwarnings('ignore')

print("All libraries imported succefully")
print("Pandas version: ", pd.__version__)
print("sqlite3 version: ", sqlite3.version)

All libraries imported succefully
Pandas version:  2.2.2
sqlite3 version:  2.6.0


In [37]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/student_performance.csv")
print(f"Dataset loaded:\n{df.shape[0]} students, {df.shape[1]} attributes")
print(f"\nColumns:\n{df.columns}")
print("\nFirst 3 rows:")
df.head(3)

Dataset loaded:
30 students, 13 attributes

Columns:
Index(['student_id', 'name', 'age', 'gender', 'department', 'semester',
       'math_score', 'science_score', 'english_score', 'programming_score',
       'attendance_percentage', 'city', 'admission_year'],
      dtype='object')

First 3 rows:


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023


In [38]:
conn = sqlite3.connect('College.db') #connection object connecting Python environment to SQLite database file College.db
cursor = conn.cursor()  #to navigate and manipulate individual rows or a database query result

# better way to connect to avoid a connection being open and locked:
# with sqlite3.connect('College.db') as conn:
#     cursor = conn.cursor()
#     cursor.execute("SELECT * FROM students")
#     results = cursor.fetchall()

df.to_sql(
    'students',     #Name of table to creare in the database
    conn,           #Which db connection object to use
    if_exists = 'replace', #Is 'students' tab;e exists, replace it
    index=False     #Do not write DataFrame row numbes as a column
)

cursor.execute("SELECT COUNT(*) FROM students")

count = cursor.fetchone()[0]
print("Database 'college.db' created successfully!")
print(f"Table students has {count} rows")


Database 'college.db' created successfully!
Table students has 30 rows


# Understanding SQL Querries
SELECT name, department FROM students WHERE department = 'Computer Science'

In [39]:
cursor.execute("SELECT name, department FROM students WHERE department = 'Computer Science'")
names = cursor.fetchall()
name_list = [n for n in names]
for n in name_list:
  print(n)

('Aarav Sharma', 'Computer Science')
('Priya Patel', 'Computer Science')
('Arjun Nair', 'Computer Science')
('Divya Singh', 'Computer Science')
('Ananya Das', 'Computer Science')
('Suresh Rao', 'Computer Science')
('Ajay Tiwari', 'Computer Science')
('Swati Kulkarni', 'Computer Science')
('Nisha Kapoor', 'Computer Science')
('Tanvi Mehta', 'Computer Science')
('Amit Bose', 'Computer Science')
('Gaurav Shukla', 'Computer Science')
('Akanksha Yadav', 'Computer Science')


In [43]:
cursor.execute("SELECT name, department, math_score, programming_score FROM students WHERE math_score>=90 AND programming_score>=90 ORDER BY (programming_score+math_score) DESC LIMIT 3")
toppers = cursor.fetchall()

for n in toppers:
  print(n[0]) #if print(n) -> prints as tuples in the format (name, department)

Ananya Das
Tanvi Mehta
Arjun Nair


In [82]:
#helper function
def run_query(sql, description="student"):
  '''
  Runs a SQL query ans returns as a pandas Dataframe.

  Paramters:
  ----------
  sql: str - The SQL query string to execute
  description : str - A label to print above results

  Returns:
  ---------
  Pandas Dataframe containing the query results
  '''
  if description:
    print(f"\n{'='*55}")
    print(f"{description}")
    print(f"{'='*55}")

  result=pd.read_sql_query(sql, conn)
  print(result.to_string(index=False))
  return result

print("Helper function 'run_query' defined successfully")
print("Usage: run_query(sql_string, description_label)")

Helper function 'run_query' defined successfully
Usage: run_query(sql_string, description_label)


In [80]:
query1 = "SELECT student_id, name, department, math_score, attendance_percentage FROM students LIMIT(10)"
result = run_query(query1, "Query1: First 10 students (SELCT+LIMIT)")

query2 = "SELECT name, department, math_score FROM students ORDER BY math_score DESC LIMIT 5"
result2 = run_query(query2, "Query2: Top 5 Math Scorers (ORDER BY DESC + LIMIT)")

query3 = "SELECT name, math_score, science_score, programming_score, attendance_percentage FROM students WHERE department = 'Computer Science' ORDER BY programming_score"
result3 = run_query(query3, "Query3: Computer Science Students (WHERE filter)")

query4 = "SELECT name, department, attendance_percentage FROM students WHERE department != 'Civil' AND attendance_percentage>90 ORDER BY attendance_percentage DESC"
result4 = run_query(query4, "Query4: High Attendance (>90%) Ecluding Civil Dept")

query5 = "SELECT name, department, attendance_percentage FROM students WHERE attendance_percentage>93 AND department != 'Mechanical' ORDER BY attendance_percentage DESC LIMIT 3"
result5 = run_query(query5, "Query5: High Attendance (Top3) (>90%) Ecluding Mechanical Dept")


Query1: First 10 students (SELCT+LIMIT)
 student_id         name       department  math_score  attendance_percentage
       1001 Aarav Sharma Computer Science          85                     92
       1002  Priya Patel Computer Science          76                     87
       1003  Rohit Verma      Electronics          65                     78
       1004  Sneha Reddy       Mechanical          70                     95
       1005   Arjun Nair Computer Science          92                     90
       1006  Meera Joshi      Electronics          58                     72
       1007  Kiran Kumar            Civil          73                     85
       1008  Divya Singh Computer Science          88                     96
       1009 Rahul Mishra       Mechanical          62                     68
       1010   Ananya Das Computer Science          95                     98

Query2: Top 5 Math Scorers (ORDER BY DESC + LIMIT)
          name       department  math_score
    Ananya Das C

Case sensitivity: if table name is 'Departments' and query mensions 'departments' -> error
case sensitive!

In [85]:
result6 = run_query("SELECT name FROM students WHERE name BETWEEN 'A' AND 'F'", "Query6: Students names BETWEEN A and F")


Query6: Students names BETWEEN A and F
          name
  Aarav Sharma
    Arjun Nair
   Divya Singh
    Ananya Das
   Ajay Tiwari
Deepak Chauhan
     Amit Bose
Akanksha Yadav
